In [2]:
import os
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter


# =========================
# Excel保存路径
# =========================

desktop = os.path.join(
    os.path.expanduser("~"),
    "Desktop"
)

output_file = os.path.join(
    desktop,
    "快手作品数据.xlsx"
)


# 配置 Chrome 浏览器选项
options = Options()
options.add_argument(r"--user-data-dir=C:\Users\qieziclub1660ti\AppData\Local\Google\Chrome\SeleniumData")  # 替换为实际用户数据目录
options.add_argument("--start-maximized")  # 最大化窗口

# 创建 Chrome WebDriver
webdriver_path = r"C:\WebDriver\chromedriver.exe"

driver = webdriver.Chrome(
    service=Service(webdriver_path),
    options=options
)



# =========================
# 打开快手创作者中心
# =========================

url = "https://cp.kuaishou.com/article/manage/video"

try:

    driver.get(url)

    print("成功进入快手创作者中心")


except Exception as e:

    print(f"打开页面失败：{e}")

    driver.quit()

    exit()


# =========================
# 等待登录和作品列表出现
# =========================

wait = WebDriverWait(driver, 300)

try:

    print("请扫码登录快手创作者中心...")

    wait.until(
        EC.presence_of_element_located(
            (
                By.CSS_SELECTOR,
                "div.video-item__detail"
            )
        )
    )

    print("登录成功，进入视频管理页面")


except Exception as e:

    print(f"登录失败或等待超时：{e}")

    driver.quit()

    exit()


# =========================
# 数字转换函数
# =========================

def convert_number(text):

    if not text:
        return 0

    text = text.strip()

    # 去掉空格、换行及常见文字
    text = (
        text
        .replace("播放", "")
        .replace("评论", "")
        .replace("点赞", "")
        .replace("次", "")
        .replace(" ", "")
        .replace("\n", "")
    )

    if text in ["", "-", "--"]:

        return 0

    # 处理“亿”
    if "亿" in text:

        try:

            return int(
                float(
                    text.replace("亿", "")
                ) * 100000000
            )

        except ValueError:

            return 0

    # 处理“万”
    if "万" in text:

        try:

            return int(
                float(
                    text.replace("万", "")
                ) * 10000
            )

        except ValueError:

            return 0

    # 处理普通数字和逗号
    try:

        return int(
            float(
                text.replace(",", "")
            )
        )

    except ValueError:

        return 0


# =========================
# 提取变量
# =========================

all_video_data = []

# 使用“标题+日期”去重
loaded_videos = set()

last_card_count = 0
last_scroll_height = 0
no_change_count = 0


# =========================
# 自动滚动并提取全部作品
# =========================

while True:

    # 查找当前已经加载的作品
    video_cards = driver.find_elements(
        By.CSS_SELECTOR,
        "div.video-item__detail"
    )

    print(
        f"\n当前页面发现作品：{len(video_cards)}"
    )


    for card in video_cards:

        try:

            # =========================
            # 提取标题
            # =========================

            title = card.find_element(
                By.CSS_SELECTOR,
                "div.video-item__detail__row__title"
            ).text.strip()


            if not title:

                continue


            # =========================
            # 提取日期
            # =========================

            try:

                date = card.find_element(
                    By.CSS_SELECTOR,
                    "div.video-item__detail__row__date"
                ).text.strip()

            except Exception:

                date = ""


            # 去掉日期前可能存在的状态文字
            date = (
                date
                .replace("已发布", "")
                .replace("待发布", "")
                .strip()
            )


            unique_key = (
                title,
                date
            )


            if unique_key in loaded_videos:

                continue


            # =========================
            # 默认数据
            # =========================

            play_count = 0
            comment_count = 0
            like_count = 0


            # =========================
            # 提取所有数据项
            # =========================

            stat_items = card.find_elements(
                By.CSS_SELECTOR,
                "div.video-item__detail__row__label"
            )


            for stat_item in stat_items:

                try:

                    # 数据值通常是整个元素的文字
                    value_text = stat_item.text.strip()

                    image = stat_item.find_element(
                        By.TAG_NAME,
                        "img"
                    )

                    image_src = (
                        image.get_attribute("src")
                        or ""
                    ).lower()


                    # 根据图标文件名判断字段
                    if "icon-play" in image_src:

                        play_count = convert_number(
                            value_text
                        )


                    elif (
                        "icon_comment" in image_src
                        or "icon-comment" in image_src
                    ):

                        comment_count = convert_number(
                            value_text
                        )


                    elif (
                        "icon-like" in image_src
                        or "icon_like" in image_src
                    ):

                        like_count = convert_number(
                            value_text
                        )


                except Exception:

                    continue


            loaded_videos.add(
                unique_key
            )


            all_video_data.append(
                {
                    "作品名称": title,
                    "发布日期": date,
                    "播放量": play_count,
                    "评论量": comment_count,
                    "点赞量": like_count
                }
            )


            print(
                f"{title} | "
                f"日期：{date} | "
                f"播放：{play_count} | "
                f"评论：{comment_count} | "
                f"点赞：{like_count}"
            )


        except Exception as e:

            print(
                f"单个作品提取失败：{e}"
            )


    # =========================
    # 判断是否全部加载完成
    # =========================

    current_card_count = len(video_cards)

    current_scroll_height = driver.execute_script(
        "return document.documentElement.scrollHeight"
    )


    if (
        current_card_count == last_card_count
        and current_scroll_height == last_scroll_height
    ):

        no_change_count += 1

    else:

        no_change_count = 0


    last_card_count = current_card_count
    last_scroll_height = current_scroll_height


    # 连续3次滚动后均没有新增内容
    if no_change_count >= 3:

        print("\n已经加载全部快手作品")

        break


    # =========================
    # 滚动加载更多作品
    # =========================

    scroll_result = driver.execute_script("""

    let all = document.querySelectorAll("*");

    let target = null;

    let maxScroll = 0;


    for(let el of all){

        let style = window.getComputedStyle(el);

        if(
            (style.overflowY === "auto" ||
            style.overflowY === "scroll")
            &&
            el.scrollHeight > el.clientHeight
        ){

            if(el.scrollHeight > maxScroll){

                maxScroll = el.scrollHeight;

                target = el;

            }

        }

    }


    if(target){

        let before = target.scrollTop;


        target.scrollTop += target.clientHeight;


        return {

            before: before,

            after: target.scrollTop,

            height: target.scrollHeight

        };

    }


    return "没有找到滚动区域";


    """)


    print(
        "滚动结果:",
        scroll_result
    )


    time.sleep(1)


# =========================
# 关闭浏览器
# =========================

driver.quit()


# =========================
# 输出提取结果
# =========================

print(
    f"\n最终提取作品数量：{len(all_video_data)}"
)


if not all_video_data:

    print("没有提取到任何快手作品数据")

    exit()


# =========================
# 转换为DataFrame
# =========================

df = pd.DataFrame(
    all_video_data
)


number_columns = [
    "播放量",
    "评论量",
    "点赞量"
]


for column in number_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    ).fillna(0).astype(int)


# =========================
# 保存Excel
# =========================

df.to_excel(
    output_file,
    index=False,
    sheet_name="快手作品数据"
)


# =========================
# 调整Excel格式
# =========================

wb = load_workbook(
    output_file
)

ws = wb.active


# 设置列宽
ws.column_dimensions["A"].width = 45
ws.column_dimensions["B"].width = 22

for column_number in range(
    3,
    ws.max_column + 1
):

    column_letter = get_column_letter(
        column_number
    )

    ws.column_dimensions[
        column_letter
    ].width = 12


# 标题行居中
for cell in ws[1]:

    cell.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )


# 日期及数字列右对齐
for row in ws.iter_rows(
    min_row=2,
    min_col=2,
    max_col=ws.max_column,
    max_row=ws.max_row
):

    for cell in row:

        cell.alignment = Alignment(
            horizontal="right",
            vertical="center"
        )


wb.save(
    output_file
)


print(
    f"快手作品数据已保存到：{output_file}"
)


# =========================
# 自动打开Excel
# =========================

try:

    os.startfile(
        output_file
    )

except Exception as e:

    print(
        f"无法自动打开Excel：{e}"
    )

成功进入快手创作者中心
请扫码登录快手创作者中心...
登录成功，进入视频管理页面

当前页面发现作品：30
#OCS 光交换机需求爆了！ #MEMS微镜 凭什么成为 #谷歌 最优解？ #光子芯片 | 日期：2026-07-13 18:30 | 播放：3983 | 评论：2 | 点赞：12
半导体气体产业链创新成果路演圆满落幕！近距离感受产业一线的供需碰撞 #电子气体 #芯片 #半导体 #AI | 日期：2026-07-07 16:53 | 播放：4266 | 评论：2 | 点赞：12
#CPO #硅光 #算力光互联 #光子芯片 从追赶到量产，硅光元年见证国产突破。本次对话嘉宾Luceda 总裁曹如平博士深耕集成光子赛道、见证硅光从科研走向商用。听她拆解光子 EDA 与传统芯片工具核心区别，解读硅光替代铜缆、支撑 CPO 演进完整路线，剖析产业链协同瓶颈，让我们一同站在算力光互联新时代。 | 日期：2026-07-01 16:53 | 播放：4237 | 评论：2 | 点赞：12
出席IPO敲钟仪式！幻实见证9年老友从一颗胎压芯片走到上市     #琻捷电子 半导体 #芯片 #IPO | 日期：2026-06-24 18:28 | 播放：4220 | 评论：3 | 点赞：10
成立仅四年，主驱 #碳化硅模块 细分领域出货量全球第一，上车至今 #零失效。 | 日期：2026-06-22 10:27 | 播放：4458 | 评论：2 | 点赞：19
揭秘！打卡国内首条光子芯片中试线！ #芯片研发 #光子芯片 | 日期：2026-06-11 10:46 | 播放：5630 | 评论：5 | 点赞：30
#中国电子院 ：一家设计院的自我革命 #半导体 #AI | 日期：2026-06-02 10:12 | 播放：4107 | 评论：3 | 点赞：20
跳出单点局限，广立微解锁AI+EDA新战力！ #广立微 #EDA #ai | 日期：2026-05-26 18:02 | 播放：4059 | 评论：3 | 点赞：18
幻实带你走进 #无锡市集成电路学会 ，一探究竟 #芯片 #集成电路 #探访 | 日期：2026-05-11 11:52 | 播放：4052 | 评论：4 | 点赞：18
今年最硬核的化合物半导体展会，幻实替你们探了！ #九峰山论坛 #新质半导体 #芯片  #LU